In [1]:
import event_helpers
import gcsfs
import geopandas as gpd
import google.auth
import pandas as pd
import poi_geography_utils as poi_geog_utils
import world_cup_vars as wc_vars
from gtfs_curator_utils import utils

GCS_FILE_PATH = wc_vars.GCS_FILE_PATH
credentials, _ = google.auth.default()

In [2]:
STADIUM_NAME = "sofi"
stop_gdf = gpd.read_parquet(
    f"{GCS_FILE_PATH}stop_summary_{STADIUM_NAME}.parquet",
    storage_options = {"token": credentials.token}
)

In [3]:
import create_route_and_stop_event_summary as A1

routes_df = A1.import_routes_near_poi(wc_vars.event_name, "sofi")

stop_gdf = A1.prep_fct_daily_scheduled_stops(wc_vars.event_name, wc_vars.sofi_match_times).merge(
    routes_df[["feed_key", "schedule_name"]].drop_duplicates(),
    how="inner",
).pipe(A1.flag_if_stop_on_near_or_special_route, routes_df)

In [ ]:
# get non-event 
arrivals_cols = [c for c in stop_gdf.columns if "arrivals_per_hour" in c]

nonevent_arrivals_df = (
    event_helpers.aggregate_by_event_type(
        stop_gdf[(stop_gdf.is_special_route==True) & (~stop_gdf.service_date.isin(wc_vars.sofi_dates)) ],
        group_cols=[
            "schedule_name",
            "stop_id",
            "stop_name",
            "event_day",
            "day_type",
        ],
        mean_cols=arrivals_cols
    )
)

In [ ]:
arrivals_by_event = pd.concat(
    [nonevent_arrivals_df,
     stop_gdf[(stop_gdf.is_special_route==True) & (stop_gdf.service_date.isin(wc_vars.sofi_dates))]
    ], axis=0, ignore_index=True
)

In [ ]:
arrivals_by_event_long = event_helpers.make_long(
    arrivals_by_event, 
    id_vars = ["service_date", "schedule_name", "stop_id", "stop_name", "day_type"] # service_date for non_event are NaT
)

In [ ]:
# Aggregate this across number of stops for operator
# Keep average arrivals_per_hour
arrivals_by_event_long2 = (
    arrivals_by_event_long
    .groupby(["service_date", "schedule_name", "time_of_day", "day_type"], dropna=False)
    .agg({
        "arrivals_per_hour": "mean",
        "stop_id": "count"
    }).reset_index().rename(columns = {"stop_id": "n_stops"})
)

arrivals_by_event_long2 = arrivals_by_event_long2.assign(
    service_date = arrivals_by_event_long2.service_date.dt.date.fillna('nonevent').astype(str),
)

In [ ]:
import altair as alt

alt.data_transformers.enable("vegafusion")

def bar_chart_by_time_of_day(df: pd.DataFrame, color_col: str):
    
    selection = alt.selection_point(fields=[color_col], bind="legend")
    
    chart = (
        alt.Chart(df)
        .mark_bar()
        .encode(
            x="arrivals_per_hour",
            y=alt.Y("time_of_day", sort=event_helpers.TIME_OF_DAY_ORDER),
            tooltip = ["schedule_name", "day_type", "time_of_day", "arrivals_per_hour", "n_stops"],
            color=alt.Color(f"{color_col}:N", sort=event_helpers.TIME_OF_DAY_ORDER),
            opacity=alt.when(selection).then(alt.value(1)).otherwise(alt.value(0.2)),        
        )
    )
    
    return chart

### Sofi weekday midday matches

In [ ]:
time_of_day = "midday"
day_type = "weekday"

def concat_bar_chart(df: pd.DataFrame, color_col: str, 
                     time_of_day: str, day_type: str):

    filtered_sofi_dict = event_helpers.grab_matches_by_day_type(
        wc_vars.sofi_match_times, time_of_day, day_type)
    
    indiv_dates = list(filtered_sofi_dict.keys())

    selection = alt.selection_point(fields=[color_col], bind="legend")

    nonevent_chart = bar_chart_by_time_of_day(
        df[(df.service_date == "nonevent") & (df.day_type==day_type)], color_col=color_col
    ).properties(title = {
        "text": f"Typical {day_type} arrivals",
    }).add_params(selection)
    
    indiv_date_charts = [
        bar_chart_by_time_of_day(
            df[df.service_date == d], color_col=color_col
        ).properties(title = {
            "text": f"SoFi Match: {d} at {time_of_day}",
            "subtitle": f"{day_type}",
        }).add_params(selection) for d in indiv_dates
    ]    

    return [nonevent_chart] + indiv_date_charts 
    

In [ ]:
event_charts = concat_bar_chart(arrivals_by_event_long2, "time_of_day", "midday", "weekday")
alt.vconcat(*event_charts).resolve_scale(x="shared")

In [ ]:
time_of_day = "midday"
day_type = "weekday"

filtered_sofi_dict = event_helpers.grab_matches_by_day_type(
    wc_vars.sofi_match_times, time_of_day, day_type)

indiv_dates = list(filtered_sofi_dict.keys())

def make_another_bar(df, time_of_day: str, day_type: str):
    selection = alt.selection_point(fields=["schedule_name"], bind="legend")
    
    chart = alt.Chart(
        df[
            (df.time_of_day==time_of_day) & 
            (df.day_type==day_type) 
            #& (df.service_date.isin(["nonevent"] + indiv_dates))
            ]
    ).mark_bar().encode(
        x="arrivals_per_hour",
        y="service_date",
        color="schedule_name",
        opacity=alt.when(selection).then(alt.value(1)).otherwise(alt.value(0.2)),        
        tooltip=["service_date", "schedule_name", "day_type", "time_of_day", "arrivals_per_hour"]
    )

    return chart

In [ ]:
selection = alt.selection_point(fields=["schedule_name"], bind="legend")

indiv_time_of_day_charts = [
    make_another_bar(
        arrivals_by_event_long2[arrivals_by_event_long2.service_date.isin(["nonevent"] + indiv_dates)], 
        time_of_day, 
        "weekday"
    ).properties(title = {
        "text": f"SoFi Match at {time_of_day}",
        "subtitle": f"{day_type}",
    }).add_params(selection) for time_of_day in event_helpers.TIME_OF_DAY_ORDER
]    

In [ ]:
alt.vconcat(*indiv_time_of_day_charts)

In [ ]:
alt.Chart(
    arrivals_by_event_long2[arrivals_by_event_long2.service_date.isin(["nonevent"] + indiv_dates)]
).mark_rect().encode(
    x=alt.X("time_of_day", sort=event_helpers.TIME_OF_DAY_ORDER),
    y="service_date",
    color="mean(arrivals_per_hour)"
)

In [4]:
stop_gdf2 = (
    stop_gdf[stop_gdf.is_special_route==True]
    .groupby(["service_date", "schedule_name", "day_type", "event_day"])
    .agg({
        f"arrivals_per_hour_{t}": "mean" for t in event_helpers.TIME_OF_DAY_ORDER}
        )
    .reset_index()
).pipe(
    event_helpers.make_long,
    id_vars = ["service_date", "schedule_name", "day_type", "event_day"] 
)

In [22]:
import altair as alt
alt.data_transformers.enable("vegafusion")


def heatmap_one_operator(df, one_operator: str):
    chart = alt.Chart(
        df[
            (df.schedule_name==one_operator) 
            ]
    ).mark_rect().encode(
        x=alt.X("time_of_day", sort=event_helpers.TIME_OF_DAY_ORDER),
        y="monthdate(service_date):T",
        color="arrivals_per_hour",
        tooltip=["service_date", "schedule_name", "day_type", "time_of_day", "arrivals_per_hour"]
    )

    return chart


In [23]:
heatmap_one_operator(stop_gdf2, "G Trans Schedule")

alt.Chart(...)

In [24]:
for i in stop_gdf2.schedule_name.unique():
    chart = heatmap_one_operator(stop_gdf2, i).properties(title=i)
    display(chart)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

In [ ]:
alt.data_transformers.enable("vegafusion")

for i in stop_gdf2.schedule_name.unique():
    
    chart = alt.Chart(
        stop_gdf2[(stop_gdf2.schedule_name == i) & (stop_gdf2.day_type=="weekday")]
    ).mark_rect().encode(
        x=alt.X("time_of_day", sort=event_helpers.TIME_OF_DAY_ORDER),
        y="service_date:T",
        color="arrivals_per_hour"
    ).properties(title=i)

    display(chart)

In [ ]:
time_of_day = "midday"
day_type = "weekday"
def make_heatmap(df, time_of_day: str, day_type: str):
    selection = alt.selection_point(fields=["schedule_name"], bind="legend")
    
    chart = alt.Chart(
        df[
            (df.time_of_day==time_of_day) & 
            (df.day_type==day_type) 
            #& (df.service_date.isin(["nonevent"] + indiv_dates))
            ]
    ).mark_bar().encode(
        x="arrivals_per_hour",
        y="service_date",
        color="schedule_name",
        opacity=alt.when(selection).then(alt.value(1)).otherwise(alt.value(0.2)),        
        tooltip=["service_date", "schedule_name", "day_type", "time_of_day", "arrivals_per_hour"]
    )

    return chart



In [ ]:
def concat_heatmap(
    df: pd.DataFrame, color_col: str, 
    time_of_day: str, day_type: str
):

    filtered_sofi_dict = event_helpers.grab_matches_by_day_type(
        wc_vars.sofi_match_times, time_of_day, day_type)
    
    indiv_dates = list(filtered_sofi_dict.keys())

    selection = alt.selection_point(fields=[color_col], bind="legend")

    indiv_time_of_day_charts = [
        make_heatmap(
            arrivals_by_event_long2[arrivals_by_event_long2.service_date.isin(["nonevent"] + indiv_dates)], 
            time_of_day, 
            "weekday"
        ).properties(title = {
            "text": f"SoFi Match at {time_of_day}",
            "subtitle": f"{day_type}",
        }).add_params(selection) for time_of_day in event_helpers.TIME_OF_DAY_ORDER
    ]    

    return indiv_date_charts 

In [ ]:
filtered_sofi_dict = event_helpers.grab_matches_by_day_type(
    wc_vars.sofi_match_times, time_of_day, day_type)

indiv_dates = list(filtered_sofi_dict.keys())
print(indiv_dates)

alt.Chart(
    arrivals_by_event_long2[
        (arrivals_by_event_long2.time_of_day=="midday") & 
        (arrivals_by_event_long2.day_type=="weekday") & 
        (arrivals_by_event_long2.service_date.isin(["nonevent"] + indiv_dates)
        )
        ]
).mark_bar().encode(
    x="arrivals_per_hour",
    y="service_date",
    color="schedule_name"
)

In [ ]:
bar_chart_by_time_of_day(
    arrivals_by_event_long2[arrivals_by_event_long2.service_date == "nonevent"]
)